In [1]:
# ADIM 0: GOOGLE DRIVE BAĞLANTISININ SAĞLANMASI
from google.colab import drive

# Drive'ın /content/drive dizinine monte edilmesi
drive.mount('/content/drive')

print("Adım 0 Tamamlandı: Google Drive bağlantısı kuruldu ve dosyalara erişim sağlandı.")

Mounted at /content/drive
Adım 0 Tamamlandı: Google Drive bağlantısı kuruldu ve dosyalara erişim sağlandı.


In [2]:
# ADIM 1: MASKELİ BÖLGE İÇİN HASSAS METRİK HESAPLAMA FONKSİYONLARI
import torch
import numpy as np
import math
from skimage.metrics import structural_similarity as ssim
import torch.nn.functional as F

def calculate_masked_metrics(hat_output_tensor, gt_tensor, mask_tensor):
    """
    HAT modelinden çıkan görüntü ile orijinal (Ground Truth) görüntüyü karşılaştırır.
    Sadece 'mask_tensor' içindeki lekelere denk gelen pikselleri değerlendirir.
    """
    # 1. Boyutların Eşitlenmesi: HAT çıktısı orijinalden büyükse, orijinal ve maske büyütülür
    _, _, h_hat, w_hat = hat_output_tensor.shape
    _, _, h_gt, w_gt = gt_tensor.shape

    if h_hat != h_gt or w_hat != w_gt:
        gt_tensor = F.interpolate(gt_tensor, size=(h_hat, w_hat), mode='bicubic', align_corners=False)
        mask_tensor = F.interpolate(mask_tensor, size=(h_hat, w_hat), mode='nearest')

    batch_size = hat_output_tensor.shape[0]
    batch_psnr = 0.0
    batch_ssim = 0.0

    # Tensörlerin Numpy dizisine çevrilip 0-255 formatına getirilmesi
    hat_np = (hat_output_tensor.cpu().detach().numpy().transpose(0, 2, 3, 1) * 255.0).clip(0, 255).astype(np.uint8)
    gt_np = (gt_tensor.cpu().detach().numpy().transpose(0, 2, 3, 1) * 255.0).clip(0, 255).astype(np.uint8)
    mask_np = mask_tensor.cpu().detach().numpy().transpose(0, 2, 3, 1)

    for i in range(batch_size):
        m = mask_np[i] # İlgili resmin maskesi (1: leke, 0: temiz alan)

        # Maskeli PSNR
        hat_masked = hat_np[i] * m
        gt_masked = gt_np[i] * m

        mask_area = np.sum(m) * 3.0 # RGB 3 kanal
        if mask_area > 0:
            mse = np.sum((hat_masked.astype(np.float32) - gt_masked.astype(np.float32)) ** 2) / mask_area
            psnr = 50.0 if mse == 0 else 10 * math.log10((255.0 ** 2) / mse)
        else:
            psnr = 0.0

        batch_psnr += psnr

        # Maskeli SSIM (Görüntünün tamamından hesaplanıp maskeye denk gelenlerin ortalaması alınır)
        _, ssim_map = ssim(gt_np[i], hat_np[i], channel_axis=-1, data_range=255, full=True)
        masked_ssim = np.sum(ssim_map * m) / np.sum(m) if mask_area > 0 else 0.0

        batch_ssim += masked_ssim

    return batch_psnr / batch_size, batch_ssim / batch_size

print("Adım 1 Tamamlandı: Maskeli metrik fonksiyonları (PSNR/SSIM) başarıyla tanımlandı.")

Adım 1 Tamamlandı: Maskeli metrik fonksiyonları (PSNR/SSIM) başarıyla tanımlandı.


In [3]:
import glob
import os

DATASET_ROOT = "/content/dataset"
IMG_FOLDER = os.path.join(DATASET_ROOT, "img")

# Şifreli zip ihtimaline karşı parolalar
passwords = [
    "mmlab_DeepFashion_inshop",
    "mmlab_DeepFashion_consumer2shop",
    "mmlab_DeepFashion_fashionsynth",
]

# Veri seti henüz çıkarılmamışsa işlem başlatılır
if not os.path.exists(IMG_FOLDER) or len(glob.glob(os.path.join(IMG_FOLDER, '**', '*.jpg'), recursive=True)) == 0:
    print("[İŞLEM] Veri seti arşivden çıkarılıyor...")

    zip_paths = [
        "/content/drive/MyDrive/Görüntü Analizinde Derin Öğrenme Yöntemleri/DeepFashion/Img/img.zip",
        "/content/drive/MyDrive/Görüntü Analizinde Derin Öğrenme Yöntemleri/DeepFashion/Img/img_highres.zip"
    ]

    target_zip = None
    for p in zip_paths:
        if os.path.exists(p):
            target_zip = p
            print(f"[TESPİT] Zip dosyası bulundu: {p}")
            break

    if target_zip:
        os.makedirs(DATASET_ROOT, exist_ok=True)
        success = False

        # Parolalı deneme
        for pwd in passwords:
            exit_code = os.system(f'7z x "{target_zip}" -p"{pwd}" -o"{DATASET_ROOT}" -y > /dev/null 2>&1')
            if exit_code == 0:
                success = True
                break

        # Parolasız deneme
        if not success:
            os.system(f'7z x "{target_zip}" -o"{DATASET_ROOT}" -y > /dev/null 2>&1')

        found_jpgs = glob.glob(f"{DATASET_ROOT}/**/*.jpg", recursive=True)
        if found_jpgs:
            IMG_FOLDER = os.path.dirname(found_jpgs[0])
            if "WOMEN" in IMG_FOLDER: IMG_FOLDER = IMG_FOLDER.split('/WOMEN')[0]
            elif "MEN" in IMG_FOLDER: IMG_FOLDER = IMG_FOLDER.split('/MEN')[0]
            print(f"[BAŞARILI] Veri seti ana dizini ayarlandı: {IMG_FOLDER}")
        else:
            print("[HATA] Zip çıkarıldı ancak içinde .jpg dosyası bulunamadı.")
    else:
        print("[HATA] Zip dosyası Google Drive yolunda bulunamadı. Yol kontrol edilmeli.")
else:
    print("[BİLGİ] Veri seti zaten mevcut, çıkarma işlemi atlanıyor.")

[İŞLEM] Veri seti arşivden çıkarılıyor...
[TESPİT] Zip dosyası bulundu: /content/drive/MyDrive/Görüntü Analizinde Derin Öğrenme Yöntemleri/DeepFashion/Img/img.zip
[BAŞARILI] Veri seti ana dizini ayarlandı: /content/dataset/img


In [4]:
# HÜCRE 3: VERİ SETİ AYRIMI VE DİNAMİK RASTGELE RENKLİ LEKE MASKELEME
import random
import glob
import cv2
import numpy as np
import torch
import os
from torch.utils.data import Dataset, DataLoader

if 'IMG_FOLDER' not in globals():
    IMG_FOLDER = "/content/dataset/img"

all_image_paths = sorted(glob.glob(os.path.join(IMG_FOLDER, '**', '*.jpg'), recursive=True))

random.seed(42)
random.shuffle(all_image_paths)

split_index = int(len(all_image_paths) * 0.90)
train_paths = all_image_paths[:split_index]
test_paths = all_image_paths[split_index:]

class FashionInpaintingDataset(Dataset):
    def __init__(self, paths, img_size=128):
        self.img_paths = paths
        self.img_size = img_size

    def generate_paint_splatter_mask(self, h, w):
        mask = np.zeros((h, w), dtype=np.float32)
        num_splatters = random.randint(5, 15)
        mu_x, mu_y = w / 2.0, h / 2.0
        sigma_x, sigma_y = w / 5.0, h / 4.0

        for _ in range(num_splatters):
            cx = max(0, min(w - 1, int(random.gauss(mu_x, sigma_x))))
            cy = max(0, min(h - 1, int(random.gauss(mu_y, sigma_y))))
            radius = random.randint(2, 5)
            cv2.circle(mask, (cx, cy), radius, 1.0, -1)

            for _ in range(random.randint(3, 8)):
                ox = max(0, min(w-1, cx + random.randint(-radius*2, radius*2)))
                oy = max(0, min(h-1, cy + random.randint(-radius*2, radius*2)))
                cv2.circle(mask, (ox, oy), 1, 1.0, -1)
        return mask

    def __getitem__(self, index):
        try:
            path = self.img_paths[index % len(self.img_paths)]
            img = cv2.imread(path)
            if img is None: raise ValueError
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (self.img_size, self.img_size))
        except: return self.__getitem__(index + 1)

        mask = self.generate_paint_splatter_mask(self.img_size, self.img_size)
        img_input = img.copy()

        # Orijinal resme rastgele RGB renkli leke basıyoruz
        random_color = np.array([random.randint(0, 255), random.randint(0, 255), random.randint(0, 255)], dtype=np.uint8)
        img_input[mask == 1.0] = random_color

        return {'input': torch.from_numpy(img_input.transpose(2,0,1)).float()/255.0,
                'mask': torch.from_numpy(mask[np.newaxis,:,:]).float(),
                'gt': torch.from_numpy(img.transpose(2,0,1)).float()/255.0}

    def __len__(self): return len(self.img_paths)

train_dataset = FashionInpaintingDataset(train_paths, img_size=128)
train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)

test_dataset = FashionInpaintingDataset(test_paths, img_size=128)
test_dataloader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=2)

dataloader = train_dataloader
print(f"[DURUM] Veri seti RASTGELE RENKLİ lekelerle başarıyla hazırlandı.")

[DURUM] Veri seti RASTGELE RENKLİ lekelerle başarıyla hazırlandı.


In [5]:
# 1. Eski ve bozuk klasörü zorla siliyoruz
!rm -rf /content/HAT

# 2. Repoyu Colab terminali üzerinden canlı olarak indiriyoruz
!git clone https://github.com/XPixelGroup/HAT.git /content/HAT

# 3. Modelin ihtiyaç duyduğu temel kütüphaneyi kuruyoruz
!pip install basicsr -q

# --- EKLENEN KRİTİK YAMA ---
# basicsr kütüphanesindeki torchvision sürüm uyuşmazlığını (functional_tensor hatasını) otomatik çözer
!sed -i 's/from torchvision.transforms.functional_tensor import rgb_to_grayscale/from torchvision.transforms.functional import rgb_to_grayscale/g' /usr/local/lib/python*/dist-packages/basicsr/data/degradations.py
# ---------------------------

# 4. Yolları tekrar sisteme işliyoruz
import sys
if '/content/HAT' not in sys.path:
    sys.path.append('/content/HAT')
if '/content/HAT/hat/archs' not in sys.path:
    sys.path.append('/content/HAT/hat/archs')

print("\n[BAŞARILI] Sistem tamamen temizlendi, gerçek dosyalar indirildi, yama uygulandı ve yollar eklendi!")

Cloning into '/content/HAT'...
remote: Enumerating objects: 419, done.
remote: Counting objects: 100% (242/242), done.
remote: Compressing objects: 100% (122/122), done.
remote: Total 419 (delta 197), reused 120 (delta 120), pack-reused 177 (from 2)
Receiving objects: 100% (419/419), 20.72 MiB | 15.60 MiB/s, done.
Resolving deltas: 100% (232/232), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.5/172.5 kB 15.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.1/333.1 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 146.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 29.5 MB/s eta 0:00:00

[BAŞARILI] Sistem tamamen temizlendi, gerçek dosyalar indirildi, yama uygulandı ve yollar eklendi!


In [6]:
# HÜCRE 1: ORTAM KURULUMU VE BAĞLANTILAR
import os
import sys
import gc
import torch
from google.colab import drive

# 1. GPU Bellek Temizliği
gc.collect()
torch.cuda.empty_cache()

# 2. Google Drive Bağlantısı
if not os.path.exists('/content/drive'):
    print("[BİLGİ] Google Drive bağlanıyor...")
    drive.mount('/content/drive')

# 3. Gerekli Kütüphanelerin Kurulumu
print("[BİLGİ] Bağımlılıklar kuruluyor...")
os.system('pip install basicsr -q')

# basicsr torchvision sürüm hatasını çözen yama sisteme dahil edildi
os.system("sed -i 's/from torchvision.transforms.functional_tensor import rgb_to_grayscale/from torchvision.transforms.functional import rgb_to_grayscale/g' /usr/local/lib/python*/dist-packages/basicsr/data/degradations.py")

os.system('apt-get install p7zip-full -y -qq')

# 4. HAT Mimarisinin Klonlanması
if not os.path.exists('/content/HAT'):
    print("[BİLGİ] HAT deposu klonlanıyor...")
    os.system('git clone https://github.com/XPixelGroup/HAT.git /content/HAT')

# 5. Sistem Yollarının Tanımlanması
sys.path.append('/content/HAT')
sys.path.append('/content/HAT/hat/archs')

# 6. Registry Çakışması Düzeltmesi
try:
    from basicsr.utils.registry import ARCH_REGISTRY
    if hasattr(ARCH_REGISTRY, '_obj_map'): ARCH_REGISTRY._obj_map.clear()
except: pass

# 7. Donanım Kontrolü
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[SİSTEM] Kullanılan Donanım: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

[BİLGİ] Bağımlılıklar kuruluyor...
[SİSTEM] Kullanılan Donanım: NVIDIA A100-SXM4-80GB


In [7]:
import importlib.util
import torch.nn as nn

# Kayıt Defteri Çakışma Önleyici
try:
    from basicsr.utils.registry import ARCH_REGISTRY
    if hasattr(ARCH_REGISTRY, '_obj_map'): ARCH_REGISTRY._obj_map.clear()
except: pass

# Orijinal HAT Çekirdeğini Yükleme
hat_file_path = '/content/HAT/hat/archs/hat_arch.py'
spec = importlib.util.spec_from_file_location("hat_arch", hat_file_path)
hat_module = importlib.util.module_from_spec(spec)
sys.modules["hat_arch"] = hat_module
spec.loader.exec_module(hat_module)
OriginalHAT = hat_module.HAT

# Segmentasyonsuz (Maskesiz) Saf Onarım Mimarisi
class BlindHAT(nn.Module):
    def __init__(self, img_size=128):
        super(BlindHAT, self).__init__()

        # 180 kanal özellik çıkarıcı
        self.hat = OriginalHAT(img_size=img_size, embed_dim=180, upscale=1, window_size=8)
        if hasattr(self.hat, 'conv_last'): del self.hat.conv_last

        # Segmentasyon bloğu tamamen iptal edildi. 180 kanal doğrudan onarıma girer.
        self.inpaint_head = nn.Sequential(
            nn.Conv2d(180, 128, 3, 1, 1),
            nn.LeakyReLU(0.2),
            nn.Conv2d(128, 64, 3, 1, 1),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 3, 3, 1, 1),
            nn.Sigmoid()
        )

    def forward_backbone(self, x):
        x = self.hat.conv_first(x)
        shortcut = x
        if hasattr(self.hat, 'patch_embed'): x = self.hat.patch_embed(x)
        if hasattr(self.hat, 'layers'):
            x_size = (x.size(2), x.size(3)) if x.dim() == 4 else (int(x.size(1)**0.5), int(x.size(1)**0.5))
            params = {'rpi_sa': getattr(self.hat, 'relative_position_index_SA', None),
                      'rpi_oca': getattr(self.hat, 'relative_position_index_OCA', None),
                      'attn_mask': None}
            for layer in self.hat.layers:
                try: x = layer(x, x_size, params)
                except: x = layer(x, x_size)
        if hasattr(self.hat, 'norm'): x = self.hat.norm(x)
        if hasattr(self.hat, 'patch_unembed'): x = self.hat.patch_unembed(x, x_size)
        x = self.hat.conv_after_body(x)
        x = x + shortcut
        return x

    def forward(self, x):
        feat = self.forward_backbone(x)
        restored_img = self.inpaint_head(feat)
        return restored_img

print("[BİLGİ] BlindHAT mimarisi tanımlandı.")

[BİLGİ] BlindHAT mimarisi tanımlandı.


In [8]:
import math
from skimage.metrics import structural_similarity as ssim

def calculate_masked_metrics(hat_output_tensor, gt_tensor, mask_tensor):
    hat_output_tensor = torch.clamp(hat_output_tensor, 0.0, 1.0)
    batch_size = hat_output_tensor.shape[0]
    batch_psnr, batch_ssim = 0.0, 0.0

    hat_np = (hat_output_tensor.cpu().detach().numpy().transpose(0, 2, 3, 1) * 255.0).astype(np.uint8)
    gt_np = (gt_tensor.cpu().detach().numpy().transpose(0, 2, 3, 1) * 255.0).astype(np.uint8)
    mask_np = mask_tensor.cpu().detach().numpy().transpose(0, 2, 3, 1)

    for i in range(batch_size):
        m = mask_np[i]
        mask_area = np.sum(m) * 3.0

        if mask_area > 0:
            mse = np.sum(((hat_np[i]*m).astype(np.float32) - (gt_np[i]*m).astype(np.float32)) ** 2) / mask_area
            psnr = 50.0 if mse == 0 else 10 * math.log10((255.0 ** 2) / mse)

            _, ssim_map = ssim(gt_np[i], hat_np[i], channel_axis=-1, data_range=255, full=True)
            masked_ssim = np.sum(ssim_map * m) / np.sum(m)
        else:
            psnr, masked_ssim = 0.0, 0.0

        batch_psnr += psnr
        batch_ssim += masked_ssim

    return batch_psnr / batch_size, batch_ssim / batch_size

print("[BİLGİ] Metrik fonksiyonu hazır.")

[BİLGİ] Metrik fonksiyonu hazır.


In [9]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

blind_model = BlindHAT(img_size=128).to(device)
blind_path = "/content/drive/MyDrive/Goruntu_Analizi_Projesi/Modeller/blind_en_iyi_model.pth"

try:
    blind_model.load_state_dict(torch.load(blind_path, map_location=device))
    print("[BAŞARILI] Segmentasyonsuz (Blind) model ağırlıkları yüklendi.")
except Exception as e:
    print(f"[HATA] Ağırlık yüklenemedi. Lütfen dosya yolunu kontrol edin: {e}")

blind_model.eval()

print("\n[FİNAL] Segmentasyonsuz model için maskeli metrikler hesaplanıyor...\n")
total_p, total_s, count = 0.0, 0.0, 0

with torch.no_grad():
    for batch in test_dataloader:
        imgs = batch['input'].to(device)
        gts = batch['gt'].to(device)
        msks = batch['mask'].to(device)

        # Segmentasyon olmadan tek aşamalı onarım
        restored = blind_model(imgs)

        # Başarı sadece lekeye denk gelen gerçek koordinatlarda ölçülür
        p, s = calculate_masked_metrics(restored, gts, msks)

        total_p += p
        total_s += s
        count += 1

        if count % 10 == 0:
            print(f"Batch {count} Tamamlandı | Anlık Ortalama PSNR: {(total_p/count):.2f} dB")

if count > 0:
    print("\n" + "="*50)
    print(f"ABLASYON ÇALIŞMASI SONUÇLARI ({count} Batch)")
    print(f"Blind Model (Segmentasyon Yok) Maskeli PSNR: {total_p/count:.2f} dB")
    print(f"Blind Model (Segmentasyon Yok) Maskeli SSIM: {total_s/count:.4f}")
    print("="*50)

/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


[BAŞARILI] Segmentasyonsuz (Blind) model ağırlıkları yüklendi.

[FİNAL] Segmentasyonsuz model için maskeli metrikler hesaplanıyor...

Batch 10 Tamamlandı | Anlık Ortalama PSNR: 7.49 dB
Batch 20 Tamamlandı | Anlık Ortalama PSNR: 7.44 dB
Batch 30 Tamamlandı | Anlık Ortalama PSNR: 7.58 dB
Batch 40 Tamamlandı | Anlık Ortalama PSNR: 7.56 dB
Batch 50 Tamamlandı | Anlık Ortalama PSNR: 7.56 dB
Batch 60 Tamamlandı | Anlık Ortalama PSNR: 7.57 dB
Batch 70 Tamamlandı | Anlık Ortalama PSNR: 7.56 dB
Batch 80 Tamamlandı | Anlık Ortalama PSNR: 7.51 dB
Batch 90 Tamamlandı | Anlık Ortalama PSNR: 7.55 dB
Batch 100 Tamamlandı | Anlık Ortalama PSNR: 7.53 dB
Batch 110 Tamamlandı | Anlık Ortalama PSNR: 7.53 dB
Batch 120 Tamamlandı | Anlık Ortalama PSNR: 7.49 dB
Batch 130 Tamamlandı | Anlık Ortalama PSNR: 7.50 dB
Batch 140 Tamamlandı | Anlık Ortalama PSNR: 7.51 dB
Batch 150 Tamamlandı | Anlık Ortalama PSNR: 7.51 dB
Batch 160 Tamamlandı | Anlık Ortalama PSNR: 7.50 dB
Batch 170 Tamamlandı | Anlık Ortalama PSNR:

In [10]:
import torch
import numpy as np

# Sadece MSE hesaplayan bağımsız fonksiyon
def calculate_masked_mse_only(hat_output_tensor, gt_tensor, mask_tensor):
    # Çıktıların 0-1 aralığında olduğundan emin olunur
    hat_output_tensor = torch.clamp(hat_output_tensor, 0.0, 1.0)
    batch_size = hat_output_tensor.shape[0]
    batch_mse = 0.0

    # İşlemler için numpy dizilerine çevrilip 0-255 aralığına genişletilir
    hat_np = (hat_output_tensor.cpu().detach().numpy().transpose(0, 2, 3, 1) * 255.0).astype(np.uint8)
    gt_np = (gt_tensor.cpu().detach().numpy().transpose(0, 2, 3, 1) * 255.0).astype(np.uint8)
    mask_np = mask_tensor.cpu().detach().numpy().transpose(0, 2, 3, 1)

    for i in range(batch_size):
        m = mask_np[i]
        mask_area = np.sum(m) * 3.0 # RGB 3 kanal toplam alanı

        if mask_area > 0:
            # Sadece lekeye (m == 1) denk gelen piksellerin karesel farkı alınır
            mse_255 = np.sum(((hat_np[i]*m).astype(np.float32) - (gt_np[i]*m).astype(np.float32)) ** 2) / mask_area
            # Makaledeki Tablo I formatına (0-1) uygun hale getirilir
            normalized_mse = mse_255 / (255.0 ** 2)
        else:
            normalized_mse = 0.0

        batch_mse += normalized_mse

    return batch_mse / batch_size

print("\n[FİNAL] Sadece MSE için maskeli metrik hesaplaması başlatılıyor...\n")
total_m, count = 0.0, 0

# Test döngüsü sadece MSE çıkarmak için bellekteki modelle tekrar çalıştırılır
with torch.no_grad():
    for batch in test_dataloader:
        imgs = batch['input'].to(device)
        gts = batch['gt'].to(device)
        msks = batch['mask'].to(device)

        restored = blind_model(imgs)

        m = calculate_masked_mse_only(restored, gts, msks)

        total_m += m
        count += 1

        if count % 10 == 0:
            print(f"Batch {count} Tamamlandı | Anlık Ortalama MSE: {(total_m/count):.6f}")

if count > 0:
    print("\n" + "="*50)
    print(f"ABLASYON ÇALIŞMASI - SADECE MSE SONUCU ({count} Batch)")
    print(f"Blind Model (Segmentasyon Yok) Maskeli MSE: {total_m/count:.6f}")
    print("="*50)


[FİNAL] Sadece MSE için maskeli metrik hesaplaması başlatılıyor...

Batch 10 Tamamlandı | Anlık Ortalama MSE: 0.194668
Batch 20 Tamamlandı | Anlık Ortalama MSE: 0.194022
Batch 30 Tamamlandı | Anlık Ortalama MSE: 0.202575
Batch 40 Tamamlandı | Anlık Ortalama MSE: 0.201076
Batch 50 Tamamlandı | Anlık Ortalama MSE: 0.201117
Batch 60 Tamamlandı | Anlık Ortalama MSE: 0.201578
Batch 70 Tamamlandı | Anlık Ortalama MSE: 0.200409
Batch 80 Tamamlandı | Anlık Ortalama MSE: 0.198356
Batch 90 Tamamlandı | Anlık Ortalama MSE: 0.196360
Batch 100 Tamamlandı | Anlık Ortalama MSE: 0.198267
Batch 110 Tamamlandı | Anlık Ortalama MSE: 0.198634
Batch 120 Tamamlandı | Anlık Ortalama MSE: 0.198433
Batch 130 Tamamlandı | Anlık Ortalama MSE: 0.197098
Batch 140 Tamamlandı | Anlık Ortalama MSE: 0.197671
Batch 150 Tamamlandı | Anlık Ortalama MSE: 0.198661
Batch 160 Tamamlandı | Anlık Ortalama MSE: 0.199080
Batch 170 Tamamlandı | Anlık Ortalama MSE: 0.199517
Batch 180 Tamamlandı | Anlık Ortalama MSE: 0.198866
Batc